In [56]:
import os
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
import cv2
import numpy as np

In [57]:
PDF_PATH = "text_based_cv.pdf"
OUTPUT_PATH = "text_based.txt"

### PDF type detection

In [58]:
def has_text_layer(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text and len(text.strip()) > 20:
                    return True
    except Exception:
        pass
    return False


def has_images(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                if page.images and len(page.images) > 0:
                    return True
    except Exception:
        pass
    return False

def detect_pdf_type(pdf_path):
    """
    Returns:
        - 'text'        → text-based PDF
        - 'scanned'     → image-based PDF
        - 'vector_only' → unsupported (paths only)
    """
    if has_text_layer(pdf_path):
        return "text"
    if has_images(pdf_path):
        return "scanned"
    return "vector_only"



### Extract text from pdf

In [59]:
def extract_text_from_text_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


def preprocess_image(img):
    img = np.array(img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    return thresh


def extract_text_from_scanned_pdf(pdf_path):
    images = convert_from_path(pdf_path, dpi=200)
    full_text = ""

    for i, img in enumerate(images):
        processed = preprocess_image(img)

        text = pytesseract.image_to_string(
            processed,
            lang="eng",
            config="--oem 3 --psm 6"
        )

        print(f"OCR page {i+1} chars:", len(text))
        full_text += text + "\n"

    return full_text



In [60]:
def extract_cv_text(pdf_path):
    pdf_type = detect_pdf_type(pdf_path)
    print("Detected PDF type:", pdf_type)

    if pdf_type == "text":
        print("→ Using text extraction")
        return extract_text_from_text_pdf(pdf_path)

    if pdf_type == "scanned":
        print("→ Using OCR")
        return extract_text_from_scanned_pdf(pdf_path)

    if pdf_type == "vector_only":
        raise ValueError(
            "Unsupported PDF format: vector-only PDF "
            "(no text layer, no images)."
        )


In [61]:
if __name__ == "__main__":
    if not os.path.exists(PDF_PATH):
        raise FileNotFoundError("PDF file not found")

    try:
        text = extract_cv_text(PDF_PATH)

        if not text.strip():
            print("⚠️ Extraction completed but no readable text found")

        with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
            f.write(text)

        print("✅ Done")

    except Exception as e:
        print("❌ Error:", str(e))


Detected PDF type: text
→ Using text extraction
✅ Done
